# Lecture 05: Analytics & Window Functions
This notebook details how to configure Window Specifications, define partitioning and ordering constraints, apply row/range frame offsets, and execute analytical ranking, offset lookups, and running aggregates.

### 1. Initialize SparkSession
Setting up local master.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lecture05_WindowFunctions") \
    .master("local[2]") \
    .getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark Session initialized successfully!")

### 2. Setup Mock Sales & Revenue Dataset
We construct a dataset representing employee sales across different departments.

In [ ]:
sales_data = [
    ("Sales", "Alice", 5000, "2026-06-01"),
    ("Sales", "Bob", 7000, "2026-06-02"),
    ("Sales", "Charlie", 5000, "2026-06-03"),
    ("IT", "Dave", 6000, "2026-06-01"),
    ("IT", "Eve", 9000, "2026-06-02"),
    ("IT", "Frank", 6000, "2026-06-03"),
    ("HR", "Grace", 4000, "2026-06-01"),
    ("HR", "Heidi", 4500, "2026-06-02")
]
df = spark.createDataFrame(sales_data, ["department", "name", "revenue", "date"])
df.show()

### 3. Window Spec: PartitionBy
Defining a window specification partitioned by department. This isolates computations inside each department group.

In [ ]:
from pyspark.sql.window import Window

w_dept = Window.partitionBy("department")
print("Window partition spec initialized.")

### 4. Window Spec: OrderBy
Adding sorting constraints inside the partition context.

In [ ]:
from pyspark.sql.functions import col

w_dept_ordered = w_dept.orderBy(col("revenue").desc())
print("Ordered window spec initialized.")

### 5. Row-based Frame Bounds (rowsBetween)
Configuring frame boundaries relative to physical row index offsets (e.g. from the start of the partition to the current row).

In [ ]:
w_rows = w_dept_ordered.rowsBetween(Window.unboundedPreceding, Window.currentRow)
print("Row-based boundary spec initialized.")

### 6. Value-based Frame Bounds (rangeBetween)
Defining frame boundaries based on numeric column value thresholds rather than row indexes.

In [ ]:
w_range = w_dept.orderBy("revenue").rangeBetween(-1000, 1000)
print("Range-based value spec initialized.")

### 7. Analytical Ranking: row_number()
Assigning unique sequential row indices to each row within a partition.

In [ ]:
from pyspark.sql.functions import row_number

df.withColumn("row_num", row_number().over(w_dept_ordered)).show()

### 8. Analytical Ranking: rank()
Ranking values. If values are identical, they receive the same rank, and subsequent ranks are skipped.

In [ ]:
from pyspark.sql.functions import rank

df.withColumn("rank", rank().over(w_dept_ordered)).show()

### 9. Analytical Ranking: dense_rank()
Similar to rank(), but duplicate values receive the same rank without skipping subsequent ranks.

In [ ]:
from pyspark.sql.functions import dense_rank

df.withColumn("dense_rank", dense_rank().over(w_dept_ordered)).show()

### 10. Analytical Ranking: percent_rank()
Calculating the relative percentile rank index.

In [ ]:
from pyspark.sql.functions import percent_rank

df.withColumn("percent_rank", percent_rank().over(w_dept_ordered)).show()

### 11. Offset Lookups: lag()
Fetching data from a preceding row at a specified offset. Useful for comparing changes over time.

In [ ]:
from pyspark.sql.functions import lag

w_time = Window.partitionBy("department").orderBy("date")
df.withColumn("prev_day_revenue", lag("revenue", 1).over(w_time)).show()

### 12. Offset Lookups: lead()
Fetching data from a succeeding row at a specified offset.

In [ ]:
from pyspark.sql.functions import lead

df.withColumn("next_day_revenue", lead("revenue", 1).over(w_time)).show()

### 13. Boundary Fetch: first()
Retrieving the first value in the partition scope.

In [ ]:
from pyspark.sql.functions import first

df.withColumn("first_revenue", first("revenue").over(w_dept_ordered)).show()

### 14. Boundary Fetch: last()
Retrieving the last value in the partition scope. This requires expanding the frame to include following rows, as the default frame stops at the current row.

In [ ]:
from pyspark.sql.functions import last

w_unbounded_following = w_dept_ordered.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
df.withColumn("last_revenue", last("revenue").over(w_unbounded_following)).show()

### 15. Running Totals (Cumulative Sums)
Calculating the cumulative sum of revenue over an unbounded preceding frame.

In [ ]:
from pyspark.sql.functions import sum

df.withColumn("cumulative_revenue", sum("revenue").over(w_rows)).show()

### 16. Rolling Moving Averages
Computing the average revenue over a sliding window (the current row and the preceding row).

In [ ]:
from pyspark.sql.functions import avg

w_moving = Window.partitionBy("department").orderBy("date").rowsBetween(-1, Window.currentRow)
df.withColumn("moving_average", avg("revenue").over(w_moving)).show()

### 17. Max Value per Partition
Identifying the highest sales record in each department.

In [ ]:
from pyspark.sql.functions import max

df.withColumn("max_dept_revenue", max("revenue").over(w_unbounded_following)).show()

### 18. Min Value per Partition
Identifying the lowest sales record in each department.

In [ ]:
from pyspark.sql.functions import min

df.withColumn("min_dept_revenue", min("revenue").over(w_unbounded_following)).show()

### 19. Deduplication using Row Numbers
Filtering out all records except the highest-performing sale per department.

In [ ]:
df.withColumn("rn", row_number().over(w_dept_ordered)) \
  .filter(col("rn") == 1) \
  .drop("rn") \
  .show()

### 20. Combining Multiple Window Specs
Applying different ranking and aggregations in a single query.

In [ ]:
df.select(
    col("department"),
    col("name"),
    col("revenue"),
    row_number().over(w_dept_ordered).alias("rank"),
    sum("revenue").over(w_rows).alias("running_sum"),
    avg("revenue").over(w_moving).alias("moving_avg")
).show()

### 21. Stop Spark Session
Stopping resources cleanly.

In [ ]:
spark.stop()